# Simulación basada en agentes con Mesa

## Caso: adopción de un canal digital en una mesa de servicio administrativo

Una organización tiene una mesa de servicio que atiende solicitudes internas. Actualmente, muchas personas utilizan correo o atención manual. El área desea impulsar un portal digital porque permite registrar mejor las solicitudes, dar seguimiento y reducir el tiempo de resolución.

Sin embargo, la adopción no ocurre de manera uniforme:

- algunas personas prueban el portal rápidamente;
- otras esperan a ver si sus colegas tienen una buena experiencia;
- algunas son resistentes al cambio;
- la capacitación puede acelerar la adopción;
- los empleados promotores pueden influir más que los usuarios comunes.

La pregunta que se desea resolver es:

> ¿Qué combinación de capacitación y empleados promotores puede lograr una adopción suficiente del canal digital sin suponer que todas las personas se comportan igual?

Este notebook usa Mesa para representar empleados como agentes autónomos que interactúan durante 30 días.

## tl;dr

La simulación basada en agentes estudia cómo reglas sencillas de personas individuales pueden producir un comportamiento colectivo.

En este caso:

- cada empleado es un agente;
- cada agente pertenece a un departamento;
- los agentes observan a colegas cercanos;
- algunos son promotores iniciales;
- la capacitación cambia la probabilidad de adoptar;
- quienes adoptan tienen mayor probabilidad de enviar solicitudes por el canal digital.

Se compararán cuatro escenarios:

1. sin capacitación y pocos promotores;
2. capacitación moderada y pocos promotores;
3. sin capacitación y más promotores;
4. capacitación moderada y más promotores.

Se observará:

- porcentaje de adopción por día;
- adopción por departamento;
- proporción de solicitudes digitales;
- tiempo promedio de resolución;
- variabilidad entre réplicas.

La recomendación final se hará considerando adopción, calidad del servicio y esfuerzo de implementación.

## 1. ¿Qué es la simulación basada en agentes?

Una simulación basada en agentes representa entidades individuales que:

1. tienen atributos;
2. toman decisiones;
3. interactúan con otras entidades;
4. cambian su comportamiento con el tiempo.

Los agentes pueden ser clientes, empleados, hogares, máquinas, vehículos o departamentos. La idea central es que el resultado global no se impone directamente: emerge de muchas decisiones individuales.

### Diferencia con otros métodos

En un promedio estadístico se podría decir que 60% de la organización adopta el portal. En un modelo basado en agentes preguntamos cómo se llega a ese 60%:

- ¿la adopción comenzó en todos los departamentos?;
- ¿se concentró en un solo grupo?;
- ¿los promotores conectan grupos distintos?;
- ¿la capacitación ayudó a las personas resistentes?;
- ¿existen departamentos que se quedan atrás?

Mesa proporciona estructuras para crear agentes, gestionar el tiempo, organizar interacciones y recoger datos. El modelo de este notebook usa una red sencilla de compañeros: cada empleado interactúa con colegas de su mismo departamento.

## 2. Contexto operativo del servicio

El portal digital no es solamente una herramienta tecnológica. Cambia el proceso de servicio:

- el correo y la atención manual requieren más registro y seguimiento;
- el portal captura información estructurada;
- una solicitud digital puede asignarse y rastrearse con mayor facilidad;
- una adopción baja mantiene trabajo manual para la mesa de servicio;
- una adopción alta puede reducir tiempos de resolución.

Para representar esta relación se usarán tiempos de resolución plausibles:

| Canal | Mínimo | Típico | Máximo |
|---|---:|---:|---:|
| Digital | 1 día | 2 días | 4 días |
| Manual/correo | 3 días | 5 días | 10 días |

Los tiempos están expresados en días hábiles aproximados. No son mediciones reales de una organización; son supuestos didácticos que permiten estudiar el efecto del canal.

Cada agente tendrá una probabilidad diaria de generar una solicitud. Si ya adoptó el portal, tendrá una probabilidad mayor de utilizarlo.

## 3. Reglas del modelo

Cada día ocurre lo siguiente:

1. Se observa el estado de adopción de los colegas.
2. Las personas no adoptantes calculan una probabilidad de adoptar.
3. Esa probabilidad aumenta con la influencia de colegas adoptantes.
4. La capacitación agrega una influencia positiva.
5. Las personas resistentes tienen una probabilidad menor.
6. Las personas adoptantes pueden convertirse en promotoras.
7. Cada empleado puede generar una solicitud de servicio.
8. El canal elegido determina una distribución de tiempo de resolución.
9. Mesa guarda los resultados diarios.

La adopción no es una certeza. Una persona con muchos colegas adoptantes puede seguir sin adoptar ese día. Esto representa que los seres humanos tienen preferencias, dudas, prioridades y restricciones.

## 4. Parámetros y supuestos de realismo

Se simularán 120 empleados durante 30 días. Se repartirán entre cuatro departamentos, con 30 personas cada uno.

Parámetros principales:

- 8% de adoptantes iniciales;
- 5% de promotores iniciales;
- 6 colegas por agente;
- probabilidad diaria de generar una solicitud: 12%;
- probabilidad de usar el canal digital si ya se adoptó: 80%;
- probabilidad de usar el canal digital si no se adoptó: 15%;
- capacitación moderada: 60% de cobertura.

Los porcentajes no pretenden describir una empresa específica. Son valores de demostración suficientemente pequeños para que la adopción pueda crecer gradualmente y suficientemente grandes para que las interacciones produzcan un patrón visible.

Se fijará una semilla para que los resultados sean reproducibles.

## 5. Instalar e importar librerías

Mesa es el marco de simulación basada en agentes. Se instalará en Colab junto con seaborn, que facilita los gráficos estadísticos.

También se importan:

- numpy para cálculos;
- pandas para tablas;
- matplotlib y seaborn para visualizaciones;
- dataclass para definir agentes de manera legible;
- networkx para dibujar una red de interacción en una jornada inicial.

La red se usa como una herramienta didáctica: ayuda a visualizar quién puede influir en quién.

In [ ]:
!pip install -q mesa seaborn networkx

import mesa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from dataclasses import dataclass
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Mesa:", mesa.__version__)
print("Librerías cargadas correctamente.")

### Interpretación del bloque de instalación

La instalación es necesaria porque Google Colab comienza con un entorno que puede no tener Mesa disponible.

La versión se imprime para dejar evidencia de qué API se utilizó. Esto es útil porque las librerías evolucionan y una diferencia de versión puede cambiar algunos nombres o comportamientos.

El modelo se ha diseñado con las clases principales de Mesa: Model y Agent, y con el conjunto de agentes administrado por el modelo.

## 6. Definir los parámetros del experimento

Centralizar los parámetros permite cambiar la organización simulada sin buscar números escondidos dentro de las funciones.

El parámetro de capacitación se interpreta como cobertura: si vale 0.60, aproximadamente 60% de los empleados recibe apoyo o capacitación. No significa que 60% adopte automáticamente.

El parámetro de influencia representa el efecto de observar a colegas adoptantes. Se mantiene moderado para que la influencia social ayude, pero no domine por completo la decisión individual.

In [ ]:
SEMILLA_BASE = 2026
POBLACION = 120
DEPARTAMENTOS = ["Finanzas", "Recursos Humanos", "Operaciones", "Compras"]
DIAS = 30

ADOPCION_INICIAL = 0.08
PROMOTORES_INICIALES = 0.05
COLEGAS_POR_AGENTE = 6

PROB_SOLICITUD_DIARIA = 0.12
PROB_DIGITAL_ADOPTANTE = 0.80
PROB_DIGITAL_NO_ADOPTANTE = 0.15

MIN_DIGITAL, MODA_DIGITAL, MAX_DIGITAL = 1.0, 2.0, 4.0
MIN_MANUAL, MODA_MANUAL, MAX_MANUAL = 3.0, 5.0, 10.0

INFLUENCIA_PARES = 0.18
BONO_CAPACITACION = 0.06
PENALIZACION_RESISTENCIA = 0.35
PROB_CONVERTIRSE_PROMOTOR = 0.03

print(f"Población: {POBLACION} agentes")
print(f"Duración: {DIAS} días")
print(f"Departamentos: {len(DEPARTAMENTOS)}")

### Revisión de unidades y magnitudes

Los días son la unidad del tiempo de simulación. Los tiempos de resolución también están en días, por lo que no se mezclan horas, minutos y días dentro del modelo.

La probabilidad diaria de solicitud de 0.12 significa que cada empleado tiene, en promedio, 12% de probabilidad de generar una solicitud en un día. Para 120 empleados, se esperan aproximadamente 14 solicitudes diarias, aunque la cantidad real cambia por la aleatoriedad.

Los tiempos digitales de 1 a 4 días y manuales de 3 a 10 días hacen visible una diferencia operativa razonable: el canal digital no elimina el trabajo, pero puede reducir captura, retrabajo y seguimiento manual.

Todos los parámetros están entre rangos válidos: los porcentajes están entre 0 y 1 y los tiempos son positivos.

## 7. Definir los estados de cada agente

Cada empleado tendrá:

- un identificador;
- un departamento;
- un estado de adopción;
- una marca de resistencia;
- una marca de promotor;
- una lista de colegas con quienes interactúa.

Los estados principales son:

- no adoptante;
- adoptante;
- resistente.

Un promotor también es adoptante, pero tiene una capacidad de influencia mayor en la interpretación del escenario. La resistencia no significa que la persona sea negativa: representa mayor fricción, dudas o preferencia por el proceso anterior.

In [ ]:
@dataclass
class RegistroSolicitud:
    dia: int
    agente_id: int
    departamento: str
    canal: str
    tiempo_resolucion_dias: float

### Explicación del registro de solicitudes

Esta estructura guarda una fila por solicitud de servicio. El agente que genera la solicitud sigue siendo la unidad individual del modelo, pero el registro permite analizar el proceso de servicio.

Se guarda el departamento para comparar si la adopción se distribuye de manera equilibrada. También se guarda el canal porque la decisión de utilizar el portal es el vínculo entre comportamiento individual y desempeño del servicio.

## 8. Crear la clase de agente

La clase EmployeeAgent representa a una persona empleada.

Mesa registra cada nuevo agente dentro del modelo. La función step será llamada una vez por día.

El agente no conoce toda la organización. Solo observa el estado de sus colegas. Esta información limitada hace que el comportamiento sea más realista que entregar a todos los empleados el porcentaje total de adopción.

In [ ]:
class EmployeeAgent(mesa.Agent):
    def __init__(
        self,
        model,
        departamento,
        adoptante=False,
        resistente=False,
        promotor=False
    ):
        super().__init__(model)
        self.departamento = departamento
        self.adoptante = adoptante
        self.resistente = resistente
        self.promotor = promotor
        self.colegas = []

    def tasa_adopcion_colegas(self):
        if not self.colegas:
            return 0.0

        estados = [
            self.model.estado_anterior[colega.unique_id]
            for colega in self.colegas
        ]
        return sum(estados) / len(estados)

    def paso_adopcion(self):
        if self.adoptante:
            return

        influencia = self.tasa_adopcion_colegas()
        probabilidad = (
            0.01
            + self.model.influencia_pares * influencia
            + self.model.bono_capacitacion
        )

        if self.resistente:
            probabilidad *= self.model.penalizacion_resistencia

        probabilidad = min(max(probabilidad, 0.0), 1.0)

        if self.random.random() < probabilidad:
            self.adoptante = True

        if (
            self.adoptante
            and not self.promotor
            and self.random.random() < self.model.prob_promotor
        ):
            self.promotor = True

    def paso_solicitud(self):
        if self.random.random() >= self.model.prob_solicitud:
            return

        prob_digital = (
            self.model.prob_digital_adoptante
            if self.adoptante
            else self.model.prob_digital_no_adoptante
        )

        canal = (
            "Digital"
            if self.random.random() < prob_digital
            else "Manual/correo"
        )

        if canal == "Digital":
            tiempo = self.random.triangular(
                MIN_DIGITAL, MAX_DIGITAL, MODA_DIGITAL
            )
        else:
            tiempo = self.random.triangular(
                MIN_MANUAL, MAX_MANUAL, MODA_MANUAL
            )

        self.model.solicitudes.append(
            RegistroSolicitud(
                dia=self.model.dia,
                agente_id=self.unique_id,
                departamento=self.departamento,
                canal=canal,
                tiempo_resolucion_dias=tiempo
            )
        )

    def step(self):
        self.paso_adopcion()
        self.paso_solicitud()

### Explicación detallada de la clase de agente

La función tasa_adopcion_colegas calcula la proporción de colegas que ya adoptaron el canal. Si 4 de 6 colegas adoptaron, la influencia observada es 4/6.

La función paso_adopcion se ocupa del cambio de comportamiento. La probabilidad comienza con una base pequeña, aumenta con la influencia de los pares y recibe un bono si existe capacitación. En una persona resistente se reduce la probabilidad.

La función paso_solicitud representa el uso del servicio. Cada día puede no existir una solicitud; si existe, la elección del canal depende del estado de adopción. Finalmente, el canal determina el tiempo de resolución.

La función step reúne las dos decisiones diarias. Cada agente puede adoptar y también generar una solicitud. El orden y las reglas están visibles para que el modelo sea fácil de revisar.

## 9. Crear la clase de modelo

La clase ServiceAdoptionModel representa la organización completa.

El modelo se encarga de:

- crear la población;
- distribuir agentes entre departamentos;
- seleccionar adoptantes, resistentes y promotores iniciales;
- conectar a cada agente con colegas;
- ejecutar un día;
- recolectar indicadores globales.

El modelo utiliza el generador aleatorio de Mesa. Al proporcionar una semilla, las réplicas pueden reproducirse.

In [ ]:
class ServiceAdoptionModel(mesa.Model):
    def __init__(
        self,
        n_agents=POBLACION,
        dias=DIAS,
        cobertura_capacitacion=0.0,
        promotores_iniciales=PROMOTORES_INICIALES,
        seed=SEMILLA_BASE
    ):
        super().__init__(rng=seed)

        self.dias_totales = dias
        self.dia = 0
        self.cobertura_capacitacion = cobertura_capacitacion
        self.influencia_pares = INFLUENCIA_PARES
        self.bono_capacitacion = (
            BONO_CAPACITACION * cobertura_capacitacion
        )
        self.penalizacion_resistencia = PENALIZACION_RESISTENCIA
        self.prob_promotor = PROB_CONVERTIRSE_PROMOTOR
        self.prob_solicitud = PROB_SOLICITUD_DIARIA
        self.prob_digital_adoptante = PROB_DIGITAL_ADOPTANTE
        self.prob_digital_no_adoptante = PROB_DIGITAL_NO_ADOPTANTE
        self.solicitudes = []
        self.historial = []

        for i in range(n_agents):
            departamento = DEPARTAMENTOS[i % len(DEPARTAMENTOS)]
            resistente = self.random.random() < 0.15
            promotor = self.random.random() < promotores_iniciales
            adoptante = (
                self.random.random() < ADOPCION_INICIAL
                or promotor
            )

            EmployeeAgent(
                self,
                departamento=departamento,
                adoptante=adoptante,
                resistente=resistente,
                promotor=promotor
            )

        agentes = list(self.agents)
        for agente in agentes:
            posibles = [
                colega for colega in agentes
                if colega is not agente
                and colega.departamento == agente.departamento
            ]
            cantidad = min(COLEGAS_POR_AGENTE, len(posibles))
            agente.colegas = self.random.sample(posibles, cantidad)

    def registrar_dia(self):
        agentes = list(self.agents)
        total = len(agentes)
        adoptantes = sum(a.adoptante for a in agentes)
        promotores = sum(a.promotor for a in agentes)
        resistentes = sum(a.resistente for a in agentes)

        solicitudes_dia = [
            s for s in self.solicitudes if s.dia == self.dia
        ]
        digitales = sum(
            s.canal == "Digital" for s in solicitudes_dia
        )

        self.historial.append({
            "dia": self.dia,
            "adopcion_pct": 100 * adoptantes / total,
            "promotores": promotores,
            "resistentes": resistentes,
            "solicitudes": len(solicitudes_dia),
            "digitales": digitales,
            "digital_pct": (
                100 * digitales / len(solicitudes_dia)
                if solicitudes_dia else np.nan
            )
        })

    def step(self):
        self.dia += 1
        self.estado_anterior = {
            agente.unique_id: int(agente.adoptante)
            for agente in self.agents
        }
        self.agents.shuffle_do("step")
        self.registrar_dia()

    def ejecutar(self):
        for _ in range(self.dias_totales):
            self.step()

        return (
            pd.DataFrame(self.historial),
            pd.DataFrame([s.__dict__ for s in self.solicitudes])
        )

### Explicación detallada del modelo

Durante la creación, los agentes se distribuyen de forma equilibrada entre los cuatro departamentos. Se asigna una pequeña proporción inicial de adoptantes y una proporción de personas resistentes.

Después se construyen las relaciones de interacción. Cada agente observa hasta seis colegas de su mismo departamento. Esto representa que las personas suelen aprender primero de compañeros cercanos a su trabajo.

En cada día se guarda una fotografía del estado anterior. Así, la decisión de adopción usa el comportamiento que existía al inicio del día y no depende del orden accidental en que Mesa active a los agentes.

El método ejecutar repite el paso diario y devuelve dos tablas: una de indicadores de adopción y otra de solicitudes de servicio.

## 10. Primera corrida del modelo

Se ejecutará una organización durante 30 días sin capacitación adicional y con pocos promotores.

Esta corrida sirve para observar el mecanismo antes de comparar escenarios. La pregunta no es todavía cuál alternativa es mejor, sino si el modelo produce una trayectoria entendible:

- la adopción inicia baja;
- algunos agentes cambian;
- la influencia puede acelerar el crecimiento;
- el uso digital cambia conforme aumenta la adopción.

In [ ]:
modelo_prueba = ServiceAdoptionModel(
    cobertura_capacitacion=0.0,
    promotores_iniciales=0.05,
    seed=SEMILLA_BASE
)

historial_prueba, solicitudes_prueba = modelo_prueba.ejecutar()

display(historial_prueba.head())
display(solicitudes_prueba.head())

print(
    f"Adopción final: "
    f"{historial_prueba['adopcion_pct'].iloc[-1]:.1f}%"
)
print(f"Solicitudes generadas: {len(solicitudes_prueba)}")

### Interpretación de la primera corrida

La adopción final es el porcentaje de agentes que ya utilizan el canal al terminar el día 30. Puede variar si se cambia la semilla.

La tabla de solicitudes muestra que el servicio está conectado con el comportamiento de los agentes. Las solicitudes digitales y manuales no son generadas por una fórmula fija: dependen de la adopción individual.

Es normal que haya días con más solicitudes que otros. La probabilidad diaria de 12% genera un promedio, no una cantidad exacta.

## 11. Visualización de adopción y solicitudes

La primera gráfica muestra la adopción acumulada. La segunda muestra cómo cambia el porcentaje de solicitudes digitales.

La adopción y el uso digital no tienen que crecer exactamente al mismo ritmo. Una persona puede haber adoptado el portal y, aun así, enviar alguna solicitud por correo.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

sns.lineplot(
    data=historial_prueba,
    x="dia",
    y="adopcion_pct",
    marker="o",
    color="#2563EB",
    ax=ax[0]
)
ax[0].set_title("Adopción del canal digital")
ax[0].set_xlabel("Día")
ax[0].set_ylabel("Empleados adoptantes (%)")
ax[0].set_ylim(0, 100)

sns.lineplot(
    data=historial_prueba,
    x="dia",
    y="digital_pct",
    marker="o",
    color="#10B981",
    ax=ax[1]
)
ax[1].set_title("Solicitudes enviadas por canal digital")
ax[1].set_xlabel("Día")
ax[1].set_ylabel("Solicitudes digitales (%)")
ax[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

### Interpretación de las líneas

Una línea ascendente de adopción indica que nuevos agentes están cambiando de comportamiento. Si se aplana, la influencia restante ya no es suficiente para convencer a muchas personas o se alcanzó una resistencia estable.

El porcentaje digital puede tener saltos porque se calcula con las solicitudes de cada día. Si un día hubo pocas solicitudes, una sola solicitud puede cambiar bastante el porcentaje. Por ello, conviene interpretar su tendencia general y no cada punto aislado.

La separación entre adopción y uso digital es útil: adoptar el portal es una disposición, mientras que utilizarlo en una solicitud concreta es una decisión condicionada por el tipo de trámite o la preferencia del usuario.

## 12. Visualizar departamentos y red de interacción

Una simulación basada en agentes puede producir resultados diferentes entre grupos. Si un departamento comienza con pocos adoptantes o tiene más personas resistentes, puede avanzar más lentamente.

La red de ejemplo muestra agentes como puntos y relaciones de interacción como líneas. No pretende ser un organigrama real; es una representación sencilla de quién puede influir en quién.

In [ ]:
modelo_red = ServiceAdoptionModel(
    cobertura_capacitacion=0.0,
    promotores_iniciales=0.05,
    seed=SEMILLA_BASE
)

agentes_red = list(modelo_red.agents)
G = nx.Graph()

for agente in agentes_red:
    G.add_node(
        agente.unique_id,
        departamento=agente.departamento,
        adoptante=agente.adoptante
    )
    for colega in agente.colegas:
        G.add_edge(agente.unique_id, colega.unique_id)

colores = [
    "#10B981" if G.nodes[n]["adoptante"] else "#94A3B8"
    for n in G.nodes
]

plt.figure(figsize=(12, 8))
posiciones = nx.spring_layout(G, seed=SEMILLA_BASE, k=0.35)
nx.draw_networkx_edges(G, posiciones, alpha=0.08, width=0.5)
nx.draw_networkx_nodes(
    G, posiciones, node_color=colores,
    node_size=55, alpha=0.85
)
plt.title("Red de interacción: verde = adoptante inicial")
plt.axis("off")
plt.show()

### Interpretación de la red

Los puntos verdes representan adoptantes iniciales. Las líneas representan relaciones de interacción, no necesariamente comunicación formal.

Un adoptante conectado con muchas personas puede influir en una parte importante de su entorno. En cambio, un departamento con pocos enlaces hacia promotores puede avanzar lentamente aunque el promedio general de adopción sea alto.

La red también explica por qué dos organizaciones con el mismo porcentaje inicial pueden terminar de manera diferente: la posición de los agentes importa, no solo la cantidad.

In [ ]:
historial_departamentos = []

for departamento in DEPARTAMENTOS:
    datos = [
        agente for agente in agentes_red
        if agente.departamento == departamento
    ]
    historial_departamentos.append({
        "departamento": departamento,
        "adopcion_inicial_pct": 100 * np.mean(
            [agente.adoptante for agente in datos]
        ),
        "resistentes_pct": 100 * np.mean(
            [agente.resistente for agente in datos]
        )
    })

display(pd.DataFrame(historial_departamentos).round(2))

### Interpretación por departamento

Esta tabla permite identificar diferencias iniciales. Un departamento con mayor adopción inicial puede convertirse en un foco de demostración para sus colegas.

La resistencia no es una etiqueta de desempeño. Es un supuesto del modelo para representar diferencias en disposición al cambio. En una aplicación real debería estimarse con encuestas, historial de uso, entrevistas o comportamiento observado.

Analizar grupos evita que un promedio organizacional esconda áreas que necesitan apoyo específico.

## 13. Definir escenarios de intervención

Ahora se comparan cuatro políticas:

- Base: sin capacitación y 5% de promotores.
- Capacitación: 60% de cobertura y 5% de promotores.
- Promotores: sin capacitación y 15% de promotores.
- Combinada: capacitación y 15% de promotores.

La simulación se repetirá 20 veces por escenario. Las réplicas representan organizaciones posibles con diferentes secuencias aleatorias.

In [ ]:
escenarios = {
    "Base": {
        "cobertura_capacitacion": 0.0,
        "promotores_iniciales": 0.05
    },
    "Capacitación": {
        "cobertura_capacitacion": 0.60,
        "promotores_iniciales": 0.05
    },
    "Promotores": {
        "cobertura_capacitacion": 0.0,
        "promotores_iniciales": 0.15
    },
    "Combinada": {
        "cobertura_capacitacion": 0.60,
        "promotores_iniciales": 0.15
    }
}

resultados_escenarios = []
historiales = []

for nombre, parametros in escenarios.items():
    for replica in range(20):
        modelo = ServiceAdoptionModel(
            cobertura_capacitacion=parametros["cobertura_capacitacion"],
            promotores_iniciales=parametros["promotores_iniciales"],
            seed=SEMILLA_BASE + replica
        )

        historial, solicitudes = modelo.ejecutar()

        digitales = solicitudes[
            solicitudes["canal"] == "Digital"
        ]
        manuales = solicitudes[
            solicitudes["canal"] == "Manual/correo"
        ]

        resultados_escenarios.append({
            "escenario": nombre,
            "replica": replica + 1,
            "adopcion_final_pct": historial["adopcion_pct"].iloc[-1],
            "digital_final_pct": historial["digital_pct"].dropna().tail(5).mean(),
            "solicitudes_totales": len(solicitudes),
            "tiempo_resolucion_promedio": solicitudes[
                "tiempo_resolucion_dias"
            ].mean(),
            "tiempo_digital_promedio": (
                digitales["tiempo_resolucion_dias"].mean()
                if len(digitales) else np.nan
            ),
            "tiempo_manual_promedio": (
                manuales["tiempo_resolucion_dias"].mean()
                if len(manuales) else np.nan
            )
        })

        historial["escenario"] = nombre
        historial["replica"] = replica + 1
        historiales.append(historial)

resultados_escenarios = pd.DataFrame(resultados_escenarios)
historiales = pd.concat(historiales, ignore_index=True)

display(
    resultados_escenarios.groupby("escenario")[
        ["adopcion_final_pct", "digital_final_pct",
         "tiempo_resolucion_promedio"]
    ].mean().round(2)
)

### Explicación detallada de las réplicas

Cada réplica crea una organización nueva con el mismo tamaño y las mismas reglas, pero con una secuencia aleatoria diferente.

El promedio entre réplicas responde a la pregunta: ¿qué suele ocurrir bajo este escenario? La dispersión responde: ¿qué tan incierto es el resultado?

La variable digital_final_pct usa el promedio de los últimos cinco días. Esto reduce el efecto de un solo día con pocas solicitudes. La variable de adopción final se toma al día 30 porque es un estado del conjunto de agentes, no una muestra pequeña de solicitudes.

Los tiempos de resolución se calculan sobre solicitudes observadas. Como el canal digital tiene una distribución más corta, una mayor proporción digital debería reducir el tiempo total promedio.

## 14. Comparar la adopción final

La primera comparación se enfoca en el resultado principal del cambio: cuántos empleados adoptaron el canal al final de los 30 días.

La barra de error muestra la desviación estándar entre réplicas. Una desviación grande significa que el resultado depende más de la configuración aleatoria de la red y de las decisiones individuales.

In [ ]:
resumen_escenarios = (
    resultados_escenarios
    .groupby("escenario")
    .agg(
        adopcion_media=("adopcion_final_pct", "mean"),
        adopcion_sd=("adopcion_final_pct", "std"),
        digital_media=("digital_final_pct", "mean"),
        tiempo_total_medio=("tiempo_resolucion_promedio", "mean")
    )
    .reset_index()
)

orden = ["Base", "Capacitación", "Promotores", "Combinada"]
resumen_escenarios["escenario"] = pd.Categorical(
    resumen_escenarios["escenario"],
    categories=orden,
    ordered=True
)
resumen_escenarios = resumen_escenarios.sort_values("escenario")

plt.figure(figsize=(12, 5))
plt.bar(
    resumen_escenarios["escenario"],
    resumen_escenarios["adopcion_media"],
    yerr=resumen_escenarios["adopcion_sd"],
    color=["#94A3B8", "#2563EB", "#F59E0B", "#10B981"],
    capsize=5
)
plt.axhline(80, color="#DC2626", linestyle="--",
            label="Referencia didáctica: 80%")
plt.title("Adopción final del canal digital")
plt.xlabel("Escenario")
plt.ylabel("Empleados adoptantes (%)")
plt.ylim(0, 110)
plt.legend()
plt.show()

### Interpretación de la adopción final

La barra representa el resultado promedio. La línea roja es una referencia didáctica para preguntar si se alcanzó una adopción de 80%; no es una obligación universal.

Si la combinación de capacitación y promotores supera a las intervenciones individuales, existe complementariedad: la capacitación reduce dudas y los promotores hacen visible el beneficio en la práctica.

Las barras de error son importantes. Si dos escenarios tienen promedios parecidos pero uno tiene mayor variabilidad, ese escenario es menos predecible y puede requerir seguimiento adicional.

## 15. Evolución de la adopción en el tiempo

La trayectoria diaria permite saber si el cambio ocurre rápidamente o si requiere más tiempo.

Una adopción final alta puede esconder un inicio lento. Para la planeación del servicio, el ritmo importa: durante la transición la mesa puede tener que soportar simultáneamente canales digitales y manuales.

In [ ]:
plt.figure(figsize=(14, 6))

sns.lineplot(
    data=historiales,
    x="dia",
    y="adopcion_pct",
    hue="escenario",
    errorbar="sd",
    palette={
        "Base": "#64748B",
        "Capacitación": "#2563EB",
        "Promotores": "#F59E0B",
        "Combinada": "#10B981"
    }
)

plt.axhline(80, color="#DC2626", linestyle="--",
            label="Referencia 80%")
plt.title("Evolución de la adopción entre réplicas")
plt.xlabel("Día")
plt.ylabel("Adopción (%)")
plt.ylim(0, 105)
plt.legend(title="Escenario")
plt.show()

### Interpretación de la evolución

Las líneas muestran el comportamiento medio y las bandas muestran variación entre réplicas. Una curva que sube pronto indica que la intervención produce cambios tempranos.

Si la curva combinada se despega desde los primeros días, puede ser preferible cuando la organización necesita una transición rápida. Si todas las curvas convergen al final, quizá la diferencia principal sea el tiempo necesario para lograr el resultado, no el resultado final.

La velocidad de adopción es relevante para el servicio porque determina cuánto tiempo convivirán procedimientos manuales y digitales.

## 16. Impacto en el tiempo de resolución del servicio

La adopción tendría poco valor operativo si no cambiara el servicio. Por eso se compara el tiempo promedio de resolución total y se separan los canales.

El modelo no supone que una solicitud digital siempre sea instantánea. Usa una distribución triangular de 1 a 4 días. El canal manual puede tardar de 3 a 10 días debido a captura, aclaraciones y seguimiento.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(
    data=resultados_escenarios,
    x="escenario",
    y="tiempo_resolucion_promedio",
    hue="escenario",
    legend=False,
    order=orden,
    palette=["#94A3B8", "#2563EB", "#F59E0B", "#10B981"],
    ax=ax[0]
)
ax[0].set_title("Tiempo promedio de resolución por réplica")
ax[0].set_xlabel("Escenario")
ax[0].set_ylabel("Días")

canales = pd.melt(
    resultados_escenarios,
    id_vars=["escenario", "replica"],
    value_vars=[
        "tiempo_digital_promedio",
        "tiempo_manual_promedio"
    ],
    var_name="tipo_tiempo",
    value_name="dias"
)

canales["canal"] = canales["tipo_tiempo"].map({
    "tiempo_digital_promedio": "Digital",
    "tiempo_manual_promedio": "Manual/correo"
})

sns.boxplot(
    data=canales,
    x="escenario",
    y="dias",
    hue="canal",
    order=orden,
    ax=ax[1]
)
ax[1].set_title("Tiempos por canal")
ax[1].set_xlabel("Escenario")
ax[1].set_ylabel("Días")
ax[1].legend(title="Canal")

plt.tight_layout()
plt.show()

### Interpretación del impacto en el servicio

El primer boxplot muestra el resultado que experimenta el servicio completo. Los escenarios con mayor adopción digital deberían mostrar tiempos promedio menores porque una mayor parte de solicitudes utiliza el canal más rápido.

El segundo gráfico confirma la razón del efecto. La diferencia entre canales proviene de los supuestos del proceso: digital entre 1 y 4 días, manual entre 3 y 10 días.

Si el tiempo total no disminuye, puede significar que la adopción todavía es baja o que los tiempos de ambos canales son demasiado parecidos. En una aplicación real, esto sería una señal para revisar si el portal realmente elimina pasos o solamente cambia la forma de capturar la misma información.

## 17. Distribución de solicitudes por canal

La adopción es un comportamiento de los agentes. La proporción digital es el resultado observado cuando esos agentes generan solicitudes.

Este gráfico permite comprobar que el cambio de comportamiento se refleja en el proceso de servicio y no se queda únicamente como una variable interna del modelo.

In [ ]:
canal_resumen = (
    resultados_escenarios
    .groupby("escenario")
    .agg(
        digital_pct=("digital_final_pct", "mean"),
        adopcion_pct=("adopcion_final_pct", "mean")
    )
    .reset_index()
)

canal_largo = canal_resumen.melt(
    id_vars="escenario",
    value_vars=["adopcion_pct", "digital_pct"],
    var_name="indicador",
    value_name="porcentaje"
)

canal_largo["indicador"] = canal_largo["indicador"].map({
    "adopcion_pct": "Adopción de empleados",
    "digital_pct": "Solicitudes digitales"
})

plt.figure(figsize=(12, 6))
sns.barplot(
    data=canal_largo,
    x="escenario",
    y="porcentaje",
    hue="indicador",
    order=orden
)
plt.axhline(80, color="#DC2626", linestyle="--")
plt.title("Adopción individual y uso digital del servicio")
plt.xlabel("Escenario")
plt.ylabel("Porcentaje")
plt.ylim(0, 105)
plt.legend(title="")
plt.tight_layout()
plt.show()

### Interpretación del vínculo agente-servicio

La adopción individual y el porcentaje de solicitudes digitales deben estar relacionados, pero no tienen que ser idénticos.

Puede existir una adopción alta y un uso digital algo menor porque no todas las personas generan solicitudes en los últimos días o porque algunas situaciones todavía se resuelven por el canal manual.

Esta diferencia es importante para la evaluación. Medir solamente cuántas personas recibieron capacitación no demuestra que el servicio haya cambiado. Conviene observar también el comportamiento real del canal y el tiempo de resolución.

## 18. Conclusiones y decisiones posibles

El modelo permite construir una recomendación con tres niveles:

### Nivel 1: cambio de comportamiento

Revisar la adopción final y la velocidad de adopción.

### Nivel 2: operación del servicio

Revisar la proporción de solicitudes digitales y el tiempo promedio de resolución.

### Nivel 3: robustez

Revisar la variabilidad entre réplicas y la diferencia entre departamentos o grupos.

La mejor intervención no necesariamente es la que produce el porcentaje más alto con cualquier costo. Debe equilibrar rapidez, adopción, recursos de capacitación y capacidad de acompañamiento.

## 19. Conclusiones finales

1. La simulación basada en agentes permite representar empleados con comportamientos distintos, no solamente promedios organizacionales.

2. La adopción del canal digital puede surgir de interacciones locales: un empleado observa a colegas y modifica gradualmente su decisión.

3. Los promotores y la capacitación actúan de forma diferente. Los promotores aumentan la visibilidad social del cambio; la capacitación reduce dudas y facilita el uso.

4. La adopción tiene una consecuencia operativa: al aumentar el uso digital, una mayor proporción de solicitudes puede resolverse en menos días bajo los supuestos definidos.

5. La red de interacción importa. Dos organizaciones con el mismo porcentaje inicial pueden evolucionar de forma diferente si los promotores están concentrados o distribuidos.

6. Una recomendación razonable debe mirar adopción, uso real del canal, tiempo de resolución y variabilidad.

7. Para utilizar el modelo en una organización real se deben calibrar los parámetros con datos de uso, encuestas, tiempos de resolución, departamentos y relaciones de colaboración.

Este ejercicio muestra cómo un sistema global de servicio puede cambiar a partir de decisiones individuales e interacciones entre personas.

## 20. Limitaciones y siguientes pasos

Este es un modelo didáctico. No representa todas las condiciones de una implementación real.

Sería conveniente agregar:

- diferentes tipos de solicitudes;
- horarios y cargas de trabajo;
- usuarios con distintos niveles de habilidad digital;
- capacitación por departamento;
- promotores ubicados estratégicamente;
- abandono del canal digital;
- fallas o indisponibilidad del portal;
- costos de capacitación y soporte;
- indicadores de satisfacción;
- una red basada en colaboración real.

También se debe validar la distribución de tiempos. Los valores de 1 a 4 días para digital y 3 a 10 días para manual son supuestos plausibles para enseñar el método, pero deben reemplazarse con observaciones reales antes de usar el modelo para comprometer presupuesto o metas de servicio.


## 21. Lectura de los resultados observados

En la ejecución validada con Mesa 3.5.1, 120 agentes, 30 días y 20 réplicas por escenario, se observaron aproximadamente estos promedios:

| Escenario | Adopción final | Uso digital reciente | Tiempo medio de resolución |
|---|---:|---:|---:|
| Base | 92.62% | 73.23% | 4.00 días |
| Capacitación | 98.00% | 77.85% | 3.65 días |
| Promotores | 95.25% | 75.17% | 3.77 días |
| Combinada | 98.21% | 78.26% | 3.54 días |

### ¿Cómo se lee esta tabla?

El escenario Base ya alcanza una adopción alta porque existe influencia entre colegas y una pequeña cantidad de adoptantes iniciales. Esto no significa que la capacitación sea innecesaria: la adopción puede ser alta al día 30, pero el camino puede ser más lento y el servicio puede tardar más durante la transición.

La capacitación eleva la adopción porque reduce la fricción para aprender o probar el portal. El escenario Promotores también mejora, porque más personas comienzan como ejemplos visibles para sus colegas. La estrategia Combinada obtiene el mejor resultado promedio porque une las dos palancas.

El tiempo de resolución disminuye de aproximadamente 4.00 días en Base a 3.54 días en Combinada. La diferencia aparece porque una proporción mayor de solicitudes usa el canal digital, cuyo supuesto de resolución es de 1 a 4 días, frente a 3 a 10 días en el canal manual/correo.

### Conclusión operativa del escenario

Si la organización necesita acelerar la transición y mejorar el servicio, la estrategia combinada es la más sólida dentro de los supuestos del ejercicio. Si los recursos son limitados, puede comenzar con promotores distribuidos por departamento y capacitación dirigida a los grupos con menor adopción.

Estos resultados son una demostración reproducible. No deben interpretarse como el desempeño real de una empresa hasta reemplazar los supuestos por datos históricos de adopción, uso de canales y tiempos de resolución.
